In [61]:
from spacy.util import minibatch
from spacy.training.example import Example

from pathlib import Path
import json
import random
import subprocess

import spacy

## helpers

In [31]:
# GLOBAL PATHS
BASE_DIR = Path().resolve()
DATA_DIR = BASE_DIR / "backend" / "data"
PROJECT_MAPPING_FILE = DATA_DIR / "project_registry.json"

In [72]:
def read_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"{path} does not exist")
    return json.loads(path.read_text(encoding="utf-8"))

def convert_format(data):
    train_data = []

    for text, ann in data:
        entities = [tuple(ent) for ent in ann["entities"]]  # list -> tuple
        train_data.append((text, {"entities": entities}))   # list -> tuple

    return train_data

def get_project_config(project_name: str) -> tuple[str, list]:
    # Load mapping
    mapping = read_json(PROJECT_MAPPING_FILE).get("name_to_id", {})

    if project_name not in mapping:
        raise ValueError(f"Project '{project_name}' not found")

    project_id = mapping[project_name]

    # Load config
    config_path = DATA_DIR / project_id / "config.json"
    config = read_json(config_path)

    # Load train data
    train_data_path = DATA_DIR / project_id / "spacy_validated.json"
    train_data = convert_format(data = read_json(train_data_path))

    model_name = config.get("model")
    labels = config.get("labels", [])

    return project_id , model_name, labels , train_data


def load_spacy_model(model_name: str):
    try:
        nlp = spacy.load(model_name)
    except OSError:
        subprocess.run(
            ["python", "-m", "spacy", "download", model_name],
            check=True
        )
        nlp = spacy.load(model_name)
    return nlp


## 1- import data

In [73]:
project_id ,model_name, labels , train_data = get_project_config("californie-test")

print(model_name)
print(labels)

en_core_web_sm
{'CARDINAL': {}, 'DATE': {}, 'EVENT': {}, 'FAC': {}, 'GPE': {'CITY': {}}, 'LANGUAGE': {}, 'LAW': {}, 'LOC': {}, 'MONEY': {}, 'NORP': {}, 'ORDINAL': {}, 'ORG': {}, 'PERCENT': {}, 'PERSON': {'MAN': {}, 'WOMEN': {}}, 'PRODUCT': {}, 'QUANTITY': {}, 'TIME': {}, 'WORK_OF_ART': {}, 'PROFESSION': {}}


In [74]:
len(train_data)

45

In [75]:
train_data[0]

('John Smith works at Google.',
 {'entities': [(0, 10, 'MAN'), (20, 26, 'ORG')]})

In [76]:
train_data[0][0]

'John Smith works at Google.'

In [77]:
train_data[0][1]

{'entities': [(0, 10, 'MAN'), (20, 26, 'ORG')]}

## 2- train

In [68]:
nlp = load_spacy_model(model_name)

if 'ner' not in nlp.pipe_names:
    ner = nlp.add_pipe('ner')
else:
    ner = nlp.get_pipe('ner')


for _, annotations in train_data:
    for ent in annotations['entities']:
        if ent[2] not in ner.labels:
            ner.add_label(ent[2])


other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
with nlp.disable_pipes(*other_pipes):
    optimizer = nlp.create_optimizer()

    epochs = 50
    for epoch in range(epochs):
        random.shuffle(train_data)
        losses = {}
        batches = minibatch(train_data, size=2)
        for batch in batches:
            examples = []
            for text, annotations in batch:
                doc = nlp.make_doc(text)
                example = Example.from_dict(doc, annotations)
                examples.append(example)
            nlp.update(examples, drop=0.5, losses=losses)
        print(f'Epoch {epoch + 1}, Losses: {losses}')

Epoch 1, Losses: {'ner': np.float32(38.484703)}
Epoch 2, Losses: {'ner': np.float32(48.248806)}
Epoch 3, Losses: {'ner': np.float32(36.919563)}
Epoch 4, Losses: {'ner': np.float32(29.360619)}
Epoch 5, Losses: {'ner': np.float32(25.96947)}
Epoch 6, Losses: {'ner': np.float32(32.107227)}
Epoch 7, Losses: {'ner': np.float32(26.954279)}
Epoch 8, Losses: {'ner': np.float32(27.123642)}
Epoch 9, Losses: {'ner': np.float32(30.375542)}
Epoch 10, Losses: {'ner': np.float32(14.2710285)}
Epoch 11, Losses: {'ner': np.float32(9.776155)}
Epoch 12, Losses: {'ner': np.float32(27.666513)}
Epoch 13, Losses: {'ner': np.float32(16.291136)}
Epoch 14, Losses: {'ner': np.float32(18.978607)}
Epoch 15, Losses: {'ner': np.float32(14.506051)}
Epoch 16, Losses: {'ner': np.float32(6.3440332)}
Epoch 17, Losses: {'ner': np.float32(15.275691)}
Epoch 18, Losses: {'ner': np.float32(9.50277)}
Epoch 19, Losses: {'ner': np.float32(16.591877)}
Epoch 20, Losses: {'ner': np.float32(8.966322)}
Epoch 21, Losses: {'ner': np.floa

## 3- save

In [78]:
MODELS_PATH = BASE_DIR / "models" / project_id

if not MODELS_PATH.exists():
    MODELS_PATH.mkdir(parents=True, exist_ok=True)

TRAINED_MODEL_PATH = MODELS_PATH / 'custom_ner_model' # choose name

nlp.to_disk(TRAINED_MODEL_PATH)

## 4- test

In [86]:
trained_nlp = spacy.load(TRAINED_MODEL_PATH)

test_texts = [
    "John Smith works at Google.",
    "Sarah visited Madrid last summer.",
    "Can you give me the price for 6 desks?"
]

for text in test_texts:
    doc = trained_nlp(text)
    print(f'Text: {text}')
    print('Entities', [(ent.text, ent.label_) for ent in doc.ents])
    print()

Text: John Smith works at Google.
Entities [('John Smith', 'MAN'), ('Google', 'ORG')]

Text: Sarah visited Madrid last summer.
Entities [('Sarah', 'WOMEN'), ('Madrid', 'CITY'), ('last summer', 'DATE')]

Text: Can you give me the price for 6 desks?
Entities [('6', 'CARDINAL')]

